# 01 — Exploratory Data Analysis (EDA)

**Dataset:** Chest X-Ray Images (Pneumonia) — Kaggle  
**Task:** Binary classification — `NORMAL` vs `PNEUMONIA`

### Objectives
1. Understand dataset structure and class distribution
2. Analyze image properties (size, pixel distribution, quality)
3. Identify class imbalance and define mitigation strategy
4. Visualize representative samples from each class
5. Compare pixel statistics between NORMAL and PNEUMONIA radiographs

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import random
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import cv2
from PIL import Image

from src.utils.visualization import plot_dataset_distribution

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid', palette='deep')

DATA_ROOT = Path('../chest_xray')
SPLITS = ['train', 'val', 'test']
CLASSES = ['NORMAL', 'PNEUMONIA']
print('Data root:', DATA_ROOT.resolve())

## 1. Dataset Inventory

In [ ]:
records = []
for split in SPLITS:
    for cls in CLASSES:
        folder = DATA_ROOT / split / cls
        images = list(folder.glob('*.jpeg')) + list(folder.glob('*.jpg')) + list(folder.glob('*.png'))
        records.append({'split': split, 'class': cls, 'count': len(images)})

df_inventory = pd.DataFrame(records)
df_pivot = df_inventory.pivot(index='split', columns='class', values='count').fillna(0).astype(int)
df_pivot['TOTAL'] = df_pivot.sum(axis=1)
df_pivot['IMBALANCE_RATIO'] = (df_pivot['PNEUMONIA'] / df_pivot['NORMAL']).round(2)
print(df_pivot.to_string())

In [ ]:
# Visualize distribution across splits
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = {'NORMAL': '#66BB6A', 'PNEUMONIA': '#EF5350'}

for ax, split in zip(axes, SPLITS):
    data = df_inventory[df_inventory['split'] == split]
    bars = ax.bar(data['class'], data['count'],
                  color=[colors[c] for c in data['class']], edgecolor='black', linewidth=0.8)
    for bar, row in zip(bars, data.itertuples()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                f'{row.count:,}', ha='center', va='bottom', fontweight='bold', fontsize=11)
    ax.set_title(f'{split.upper()} Split', fontsize=13, fontweight='bold')
    ax.set_ylabel('Image Count')
    ax.set_ylim(0, 9000)
    ax.grid(axis='y', alpha=0.4)

plt.suptitle('Class Distribution per Split', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../reports/eda_class_distribution.png', bbox_inches='tight', dpi=150)
plt.show()

## 2. Image Property Analysis

In [ ]:
# Sample 200 images per class from training set and analyze properties
def analyze_images(folder: Path, n_sample: int = 200) -> pd.DataFrame:
    paths = list(folder.glob('*.jpeg')) + list(folder.glob('*.jpg'))
    sample = random.sample(paths, min(n_sample, len(paths)))
    records = []
    for p in sample:
        img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        records.append({
            'height': img.shape[0],
            'width':  img.shape[1],
            'aspect_ratio': img.shape[1] / img.shape[0],
            'mean_intensity': img.mean(),
            'std_intensity':  img.std(),
            'min_intensity':  img.min(),
            'max_intensity':  img.max(),
            'file_size_kb':   p.stat().st_size / 1024,
        })
    return pd.DataFrame(records)

random.seed(42)
df_normal    = analyze_images(DATA_ROOT / 'train' / 'NORMAL',    200)
df_pneumonia = analyze_images(DATA_ROOT / 'train' / 'PNEUMONIA', 200)
df_normal['class']    = 'NORMAL'
df_pneumonia['class'] = 'PNEUMONIA'
df_all = pd.concat([df_normal, df_pneumonia], ignore_index=True)

print('NORMAL images stats:')
print(df_normal.describe().round(2))
print('\nPNEUMONIA images stats:')
print(df_pneumonia.describe().round(2))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
palette = {'NORMAL': '#66BB6A', 'PNEUMONIA': '#EF5350'}

# Height distribution
for cls, color in palette.items():
    data = df_all[df_all['class'] == cls]['height']
    axes[0,0].hist(data, bins=30, alpha=0.6, label=cls, color=color, edgecolor='white')
axes[0,0].set_title('Image Height Distribution', fontweight='bold')
axes[0,0].set_xlabel('Height (px)'); axes[0,0].legend()

# Width distribution  
for cls, color in palette.items():
    data = df_all[df_all['class'] == cls]['width']
    axes[0,1].hist(data, bins=30, alpha=0.6, label=cls, color=color, edgecolor='white')
axes[0,1].set_title('Image Width Distribution', fontweight='bold')
axes[0,1].set_xlabel('Width (px)'); axes[0,1].legend()

# Mean intensity comparison
sns.violinplot(data=df_all, x='class', y='mean_intensity', palette=palette, ax=axes[0,2])
axes[0,2].set_title('Mean Pixel Intensity by Class', fontweight='bold')
axes[0,2].set_ylabel('Mean Intensity (0-255)')

# Std intensity comparison
sns.violinplot(data=df_all, x='class', y='std_intensity', palette=palette, ax=axes[1,0])
axes[1,0].set_title('Pixel Std Dev by Class', fontweight='bold')
axes[1,0].set_ylabel('Std Intensity')

# Scatter: mean vs std
for cls, color in palette.items():
    d = df_all[df_all['class'] == cls]
    axes[1,1].scatter(d['mean_intensity'], d['std_intensity'], alpha=0.4, s=15, label=cls, color=color)
axes[1,1].set_title('Mean vs Std Intensity', fontweight='bold')
axes[1,1].set_xlabel('Mean Intensity'); axes[1,1].set_ylabel('Std Intensity')
axes[1,1].legend()

# File size
sns.boxplot(data=df_all, x='class', y='file_size_kb', palette=palette, ax=axes[1,2])
axes[1,2].set_title('File Size Distribution (KB)', fontweight='bold')

plt.suptitle('Image Property Analysis: NORMAL vs PNEUMONIA', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../reports/eda_image_properties.png', bbox_inches='tight', dpi=150)
plt.show()

## 3. Visual Sample Gallery

In [ ]:
def load_sample_images(folder: Path, n: int = 6) -> list:
    paths = list(folder.glob('*.jpeg')) + list(folder.glob('*.jpg'))
    return random.sample(paths, min(n, len(paths)))

random.seed(7)
normal_samples    = load_sample_images(DATA_ROOT / 'train' / 'NORMAL', 6)
pneumonia_bacteria = [p for p in (DATA_ROOT / 'train' / 'PNEUMONIA').glob('*bacteria*')][:3]
pneumonia_virus    = [p for p in (DATA_ROOT / 'train' / 'PNEUMONIA').glob('*virus*')][:3]

fig = plt.figure(figsize=(18, 12))
gs = gridspec.GridSpec(3, 6, figure=fig, hspace=0.4, wspace=0.2)

all_samples = [
    (normal_samples, 'NORMAL', '#66BB6A', 0),
    (pneumonia_bacteria, 'PNEUMONIA (Bacterial)', '#EF5350', 1),
    (pneumonia_virus, 'PNEUMONIA (Viral)', '#FF7043', 2),
]

for samples, label, color, row in all_samples:
    for col, path in enumerate(samples[:6]):
        ax = fig.add_subplot(gs[row, col])
        img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        ax.imshow(img, cmap='gray', aspect='auto')
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(label, color=color, fontsize=10, fontweight='bold', rotation=0,
                         ha='right', va='center', labelpad=80)

fig.suptitle('Sample Chest X-Ray Images', fontsize=16, fontweight='bold')
plt.savefig('../reports/eda_sample_gallery.png', bbox_inches='tight', dpi=150)
plt.show()

## 4. Pixel Histogram Analysis

In [ ]:
def compute_mean_histogram(folder: Path, n_sample: int = 100) -> np.ndarray:
    """Compute mean pixel histogram over a sample of images."""
    paths = random.sample(
        list(folder.glob('*.jpeg')) + list(folder.glob('*.jpg')),
        min(n_sample, len(list(folder.glob('*.jpeg'))))
    )
    histograms = []
    for p in paths:
        img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        if img is not None:
            hist, _ = np.histogram(img, bins=256, range=(0, 255))
            histograms.append(hist / hist.sum())
    return np.mean(histograms, axis=0)

random.seed(42)
hist_normal    = compute_mean_histogram(DATA_ROOT / 'train' / 'NORMAL')
hist_pneumonia = compute_mean_histogram(DATA_ROOT / 'train' / 'PNEUMONIA')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

bins = np.arange(256)
ax1.fill_between(bins, hist_normal, alpha=0.6, color='#66BB6A', label='NORMAL')
ax1.fill_between(bins, hist_pneumonia, alpha=0.6, color='#EF5350', label='PNEUMONIA')
ax1.set_title('Mean Pixel Intensity Histogram', fontsize=13, fontweight='bold')
ax1.set_xlabel('Pixel Intensity'); ax1.set_ylabel('Normalized Frequency')
ax1.legend()

ax2.plot(bins, hist_pneumonia - hist_normal, color='#9C27B0', linewidth=1.5)
ax2.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax2.fill_between(bins, hist_pneumonia - hist_normal, 0,
                 where=(hist_pneumonia - hist_normal > 0), alpha=0.4, color='#EF5350',
                 label='Pneumonia > Normal')
ax2.fill_between(bins, hist_pneumonia - hist_normal, 0,
                 where=(hist_pneumonia - hist_normal < 0), alpha=0.4, color='#66BB6A',
                 label='Normal > Pneumonia')
ax2.set_title('Histogram Difference (Pneumonia - Normal)', fontsize=13, fontweight='bold')
ax2.set_xlabel('Pixel Intensity'); ax2.legend()

plt.suptitle('Pixel Distribution Analysis\n(Pneumonia shows higher mid-tone intensity → opacity)', 
             fontsize=12, color='gray')
plt.tight_layout()
plt.savefig('../reports/eda_pixel_histograms.png', bbox_inches='tight', dpi=150)
plt.show()

## 5. Key EDA Findings & Recommendations

| Finding | Value | Implication |
|---------|-------|-------------|
| Training class imbalance | ~2.89:1 (PNEUMONIA:NORMAL) | Use `WeightedRandomSampler` + class-weighted loss |
| Validation set size | 16 per class (32 total) | Unreliable; re-split from training data |
| Image dimensions | Variable (up to 2916×2583) | Must resize to 224×224 for CNN |
| Bacterial vs Viral pneumonia | Both in PNEUMONIA class | Different radiological patterns; future work could sub-classify |
| Mean intensity difference | Pneumonia > Normal | Expert system can use opacity threshold as a rule |
| Pixel std difference | Pneumonia shows more variation | Supports texture-based expert rule |